# Implementação no Azure Machine Learning

Este notebook conduz a execução do otimizador de rotas no Azure Machine Learning. O fluxo registra o ambiente e o cenário como ativos versionados, submete um job em computação serverless, acompanha a execução e recupera os artefatos produzidos.

A infraestrutura é mantida separadamente em `infra/azure`, para que possa ser revisada e aplicada de forma reproduzível. O notebook não cria recursos permanentes implicitamente e informa os pontos que podem gerar cobrança.

## Visão do fluxo

```text
Terraform -> Azure ML Workspace -> ambiente + ativo de dados
                                      |
                                      v
                           job serverless / sweep opcional
                                      |
                                      v
                   métricas MLflow + artefatos de resultado
```

O job principal executa o algoritmo genético, calcula a baseline gulosa, registra métricas no MLflow e produz `solution.json`, `routes_map.html`, `convergence.png`, `daily_report.md` e `history.csv`. Sem uma chave Gemini disponível no ambiente remoto, o relatório determinístico local é usado automaticamente; a otimização não é interrompida.

## 1. Provisionamento anterior ao notebook

O IaC deve ser aplicado em uma sessão autenticada do **Azure Cloud Shell**. Esses comandos são executados uma única vez, a partir da raiz do repositório:

```bash
cd infra/azure
terraform init
terraform fmt -check
terraform validate
terraform plan -out=tfplan
terraform apply tfplan
terraform output
```

Por padrão, são criados Resource Group, Storage Account, Key Vault, Log Analytics, Application Insights e Azure ML Workspace. O `cpu-cluster` é opcional porque o job principal usa serverless. Para criar também o cluster do sweep, depois de confirmar a cota regional, use `terraform apply -var="create_compute_cluster=true" -var="max_compute_nodes=1"`.

> **Cota:** serverless também consome cota do Azure ML. O job está configurado com um nó `Standard_DS2_v2`, que requer duas vCPUs disponíveis na região. Com cota zero, o workspace pode ser provisionado, mas nenhum job ou Compute Instance conseguirá executar.

## 2. Localização e verificação do projeto

No Azure ML Studio, abra **Notebooks**, clone ou carregue o repositório e execute este arquivo com um kernel Python. A célula abaixo localiza a raiz sem depender do diretório em que o servidor iniciou o kernel.

In [ ]:
from pathlib import Path

def localizar_raiz(inicio: Path) -> Path:
    inicio = inicio.resolve()
    for candidato in (inicio, *inicio.parents):
        if (candidato / 'pyproject.toml').exists() and (candidato / 'azure').is_dir():
            return candidato
    raise FileNotFoundError(
        'Raiz do projeto não encontrada. Carregue o repositório completo no Azure ML Studio.'
    )

ROOT = localizar_raiz(Path.cwd())
ARQUIVOS_OBRIGATORIOS = [
    ROOT / 'azure' / 'environment.yml',
    ROOT / 'azure' / 'data.yml',
    ROOT / 'azure' / 'job.yml',
    ROOT / 'azure' / 'run_job.py',
    ROOT / 'data' / 'deliveries.json',
]
ausentes = [str(p.relative_to(ROOT)) for p in ARQUIVOS_OBRIGATORIOS if not p.exists()]
if ausentes:
    raise FileNotFoundError(f'Arquivos obrigatórios ausentes: {ausentes}')
print('Raiz:', ROOT)
print('Estrutura necessária localizada com sucesso.')

## 3. SDK do Azure Machine Learning

A instalação ocorre no mesmo kernel do notebook. Quando os pacotes já estão disponíveis, a célula apenas informa a versão encontrada e evita uma reinstalação desnecessária.

In [ ]:
import importlib.metadata
import subprocess
import sys

try:
    importlib.metadata.version('azure-ai-ml')
except importlib.metadata.PackageNotFoundError:
    subprocess.check_call([
        sys.executable, '-m', 'pip', 'install',
        'azure-ai-ml>=1.24,<2', 'azure-identity>=1.17,<2', '--quiet'
    ])

print('azure-ai-ml:', importlib.metadata.version('azure-ai-ml'))
print('azure-identity:', importlib.metadata.version('azure-identity'))

## 4. Identificação do workspace

Copie os quatro valores apresentados por `terraform output`. Como alternativa, baixe `config.json` no menu do workspace no Azure ML Studio e coloque-o em `.azureml/config.json` na raiz do projeto. Esse arquivo é específico de cada ambiente e está ignorado pelo Git.

A prioridade de leitura é: variáveis de ambiente, `config.json` e valores preenchidos manualmente na célula. O identificador da assinatura não é uma credencial, mas nenhuma chave ou senha deve ser inserida aqui.

In [ ]:
import json
import os

# Preencha somente se não estiver usando variáveis de ambiente ou config.json.
CONFIGURACAO_MANUAL = {
    'subscription_id': '',
    'resource_group': '',
    'workspace_name': '',
    'location': 'brazilsouth',
}

config = dict(CONFIGURACAO_MANUAL)
config_path = ROOT / '.azureml' / 'config.json'
if config_path.exists():
    config.update({k: v for k, v in json.loads(config_path.read_text(encoding='utf-8')).items() if v})

config.update({
    'subscription_id': os.getenv('AZURE_SUBSCRIPTION_ID', config['subscription_id']),
    'resource_group': os.getenv('AZURE_RESOURCE_GROUP', config['resource_group']),
    'workspace_name': os.getenv('AZUREML_WORKSPACE_NAME', config['workspace_name']),
    'location': os.getenv('AZURE_LOCATION', config['location']),
})

obrigatorios = ('subscription_id', 'resource_group', 'workspace_name')
faltantes = [nome for nome in obrigatorios if not str(config.get(nome, '')).strip()]
if faltantes:
    raise ValueError(
        f'Preencha {faltantes} com os valores de terraform output ou forneça .azureml/config.json.'
    )

SUBSCRIPTION_ID = config['subscription_id']
RESOURCE_GROUP = config['resource_group']
WORKSPACE_NAME = config['workspace_name']
LOCATION = config['location']
print(f'Workspace configurado: {WORKSPACE_NAME} ({LOCATION})')

## 5. Autenticação e conexão

`DefaultAzureCredential` reutiliza a identidade disponível no ambiente Azure, a sessão do Azure CLI ou outra credencial compatível. Se nenhuma delas funcionar, o fluxo por código de dispositivo apresenta uma URL e um código para autenticação, sem armazenar senha no notebook.

In [ ]:
from azure.ai.ml import MLClient, load_data, load_environment, load_job
from azure.core.exceptions import ClientAuthenticationError
from azure.identity import DefaultAzureCredential, DeviceCodeCredential

credential = DefaultAzureCredential(exclude_interactive_browser_credential=False)
try:
    credential.get_token('https://management.azure.com/.default')
except ClientAuthenticationError:
    print('Credencial automática indisponível; iniciando autenticação por código de dispositivo.')
    credential = DeviceCodeCredential()
    credential.get_token('https://management.azure.com/.default')

client = MLClient(credential, SUBSCRIPTION_ID, RESOURCE_GROUP, WORKSPACE_NAME)
workspace = client.workspaces.get(WORKSPACE_NAME)
print('Conectado a:', workspace.name)
print('Resource Group:', RESOURCE_GROUP)
print('Localização:', workspace.location)

## 6. Validação do cenário e dos recursos

Antes de registrar ativos, o cenário é carregado pelo mesmo código usado no job. Essa verificação antecipa erros de estrutura no JSON. A lista de computes é apenas informativa: o job principal não depende de `cpu-cluster`.

In [ ]:
if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))

from hospital_routes.io import load_problem

problem = load_problem(ROOT / 'data' / 'deliveries.json')
print('Hospital:', problem.depot.name)
print('Entregas:', len(problem.deliveries))
print('Veículos:', len(problem.vehicles))
print('Computes existentes:', [compute.name for compute in client.compute.list()] or 'nenhum')

## 7. Registro do ambiente e do ativo de dados

O ambiente fixa Python 3.11 e as bibliotecas necessárias para visualização e rastreamento com MLflow. O JSON é registrado como `uri_file`. Os manifestos usam nome e versão explícitos, o que permite relacionar cada execução aos seus insumos. A operação pode ser repetida no mesmo workspace.

In [ ]:
environment = load_environment(source=ROOT / 'azure' / 'environment.yml')
registered_environment = client.environments.create_or_update(environment)

data_asset = load_data(source=ROOT / 'azure' / 'data.yml')
registered_data = client.data.create_or_update(data_asset)

print(f'Ambiente: {registered_environment.name}:{registered_environment.version}')
print(f'Dados: {registered_data.name}:{registered_data.version}')

## 8. Carregamento e revisão do job

O manifesto `job.yml` envia somente os arquivos permitidos por `.amlignore`. O cenário registrado é baixado para o nó, enquanto a pasta de artefatos é montada para escrita. A ausência do campo `compute` seleciona serverless; `resources` explicita um nó `Standard_DS2_v2`.

In [ ]:
job = load_job(source=ROOT / 'azure' / 'job.yml')
print('Nome:', job.display_name)
print('Experimento:', job.experiment_name)
print('Ambiente:', job.environment)
print('Compute:', job.compute or 'serverless')
print('Comando:', job.command)

## 9. Submissão e acompanhamento

A primeira execução pode levar mais tempo porque o Azure precisa construir o ambiente. O streaming mostra provisionamento, preparação, logs do algoritmo e status final. Erros `ClusterMinNodesExceedCoreQuota`, `LowPriorityCoreQuota` ou mensagens de insuficiência de vCPU indicam cota regional, não defeito no algoritmo.

In [ ]:
EXECUTAR_JOB = True

submitted_job = None
if EXECUTAR_JOB:
    submitted_job = client.jobs.create_or_update(job)
    print('Job:', submitted_job.name)
    print('Azure ML Studio:', submitted_job.studio_url)
    client.jobs.stream(submitted_job.name)
    final_job = client.jobs.get(submitted_job.name)
    print('Status final:', final_job.status)
    if final_job.status != 'Completed':
        raise RuntimeError(
            f'Job encerrado com status {final_job.status}. Consulte o link do Studio e os logs acima.'
        )
else:
    print('Submissão desativada. Altere EXECUTAR_JOB para True para executar.')

## 10. Download e inspeção dos resultados

A pasta nomeada `artifacts` é baixada para `outputs/azure_job`. Como o diretório `outputs` não é versionado, repetir a demonstração não altera o código-fonte.

In [ ]:
if submitted_job is None:
    raise RuntimeError('Execute primeiro a célula de submissão do job.')

DOWNLOAD_DIR = ROOT / 'outputs' / 'azure_job' / submitted_job.name
DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)
client.jobs.download(
    name=submitted_job.name,
    download_path=DOWNLOAD_DIR,
    output_name='artifacts',
)
artefatos = sorted(p for p in DOWNLOAD_DIR.rglob('*') if p.is_file())
for arquivo in artefatos:
    print(arquivo.relative_to(DOWNLOAD_DIR))

In [ ]:
import html
from IPython.display import HTML, Image, Markdown, display

def localizar_artefato(nome: str) -> Path:
    encontrados = list(DOWNLOAD_DIR.rglob(nome))
    if not encontrados:
        raise FileNotFoundError(f'Artefato não localizado: {nome}')
    return encontrados[0]

display(Image(filename=str(localizar_artefato('convergence.png'))))

map_html = localizar_artefato('routes_map.html').read_text(encoding='utf-8')
display(HTML(
    '<iframe style="width:100%;height:600px;border:1px solid #ccc" '
    f'srcdoc="{html.escape(map_html, quote=True)}"></iframe>'
))

report = localizar_artefato('daily_report.md').read_text(encoding='utf-8')
display(Markdown(report))

## 11. Sweep opcional de hiperparâmetros

O sweep testa população, crossover, mutação e elitismo, minimizando a métrica `fitness` registrada pelo MLflow. Ele permanece desativado por padrão porque executa diversas tentativas, demanda mais tempo e requer o `cpu-cluster` opcional do Terraform.

Antes de ativá-lo, confirme que o cluster existe e que a assinatura possui cota compatível. Com `max_concurrent_trials: 4` e quatro nós `Standard_DS2_v2`, o consumo simultâneo pode chegar a oito vCPUs. Para uma execução mais econômica, reduza `max_concurrent_trials` e `max_compute_nodes` para 1.

In [ ]:
EXECUTAR_SWEEP = False

if EXECUTAR_SWEEP:
    compute_names = {compute.name for compute in client.compute.list()}
    if 'cpu-cluster' not in compute_names:
        raise RuntimeError(
            'cpu-cluster não existe. Habilite create_compute_cluster no Terraform após obter cota.'
        )
    sweep = load_job(source=ROOT / 'azure' / 'sweep.yml')
    submitted_sweep = client.jobs.create_or_update(sweep)
    print('Sweep:', submitted_sweep.name)
    print('Azure ML Studio:', submitted_sweep.studio_url)
    client.jobs.stream(submitted_sweep.name)
else:
    print('Sweep não submetido. O job principal já executa o fluxo completo.')

## 12. Monitoramento e análise no Azure ML Studio

Na página do experimento, é possível consultar:

- parâmetros do algoritmo genético, como população, gerações e taxas dos operadores;
- `fitness`, que agrega o custo da rota e as penalidades das restrições;
- `total_distance_km` e o indicador `feasible`;
- curvas `best_fitness_by_generation` e `mean_fitness_by_generation`;
- código, ambiente, ativo de dados, logs e artefatos associados à execução.

Esse conjunto permite reproduzir a execução e relacionar os resultados aos parâmetros e insumos utilizados.

## 13. Encerramento e controle de custos

O serverless libera os nós ao final do job. Se o sweep tiver sido executado, confirme no Azure ML Studio que `cpu-cluster` retornou a zero nós. Desligue também a Compute Instance usada pelo notebook. Quando os experimentos terminarem e os recursos não forem mais necessários, remova-os de forma controlada no mesmo diretório em que o estado foi criado:

```bash
cd infra/azure
terraform plan -destroy
terraform destroy
```

A destruição é uma decisão do responsável pela assinatura e não é executada automaticamente pelo notebook.

## Referências operacionais

- [Configuração do SDK e `MLClient`](https://learn.microsoft.com/azure/machine-learning/how-to-configure-environment?view=azureml-api-2)
- [Jobs em computação serverless](https://learn.microsoft.com/azure/machine-learning/how-to-use-serverless-compute?view=azureml-api-2)
- [Schema de command jobs](https://learn.microsoft.com/azure/machine-learning/reference-yaml-job-command?view=azureml-api-2)
- [Schema de sweep jobs](https://learn.microsoft.com/azure/machine-learning/reference-yaml-job-sweep?view=azureml-api-2)
- [Cotas do Azure Machine Learning](https://learn.microsoft.com/azure/machine-learning/how-to-manage-quotas?view=azureml-api-2)